# 03 — Diagnose the attack and compare search budgets

Your previous Colab pilot reproduced the clean baseline, but both models returned all 32 inputs unchanged. This notebook records **the outputs of the gradient and evolutionary attack stages before CAA's final success filter**. It shows whether these candidates change, violate feature constraints, exceed the distance budget, or remain detected.

It compares **10 versus 100 evolutionary generations**, using the same 32 malicious records, models, seed, 10 gradient steps, 50 offspring and L2 radius 0.5. This is a diagnostic comparison, not new-model training or a published robust-score reproduction.

Use a **CPU runtime → Runtime → Run all**, then share the final ZIP. Earlier notebooks and documents are preserved; this uses a separate workspace. The longer search takes more time than notebook 02. A GPU allocation does not accelerate this pinned CPU pipeline.


## 1. Settings and provenance

Leave these defaults unchanged for the planned comparison. Candidate observers reuse CAA's cached evaluations and preserve its selected indices. They also verify random-number state preservation. Upstream files, constraints, preprocessing and attack acceptance rules are unchanged.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

PILOT_RECORDS = 32 #@param {type:"integer"}
ATTACK_STEPS = 10 #@param {type:"integer"}
GENERATIONS_LOW = 10 #@param {type:"integer"}
GENERATIONS_HIGH = 100 #@param {type:"integer"}
ATTACK_OFFSPRING = 50 #@param {type:"integer"}
EPSILON = 0.5 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

if GENERATIONS_HIGH <= GENERATIONS_LOW:
    raise ValueError("GENERATIONS_HIGH must exceed GENERATIONS_LOW.")

REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
REPO_REF = "main"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_Diagnostics32")).resolve()
HOST_PYTHON = sys.executable
HOST_VERSION = sys.version
if not 1 <= PILOT_RECORDS <= 407:
    raise ValueError("PILOT_RECORDS must be between 1 and 407; keep 32 for the planned pilot.")
if min(ATTACK_STEPS, GENERATIONS_LOW, GENERATIONS_HIGH, ATTACK_OFFSPRING) < 1 or EPSILON <= 0:
    raise ValueError("Attack budgets must be positive.")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} exists but is not a git checkout. Choose a new workspace.")
# Reuse existing work without git reset, checkout or pull; record exactly what ran.
for required in ["scripts/bootstrap.py", "baseline/run_diagnostics.py", "notebooks/03_Attack_Diagnostics_32.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the completed repository.")

def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("diagnostics32")
SESSION.mkdir(parents=True, exist_ok=False)
sha = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--verify", "HEAD"], text=True, capture_output=True)
project_commit = sha.stdout.strip() if sha.returncode == 0 else "uncommitted-workspace"
source_paths = [p for folder in ["baseline", "scripts"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/03_Attack_Diagnostics_32.ipynb"]
source_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths}
context = {"project_repository": REPO_URL, "project_commit": project_commit,
           "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
           "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
           "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "source_sha256": source_hashes,
           "experiment": "32_record_candidate_diagnostics", "evaluation_device": "cpu",
           "pilot_records": PILOT_RECORDS, "steps": ATTACK_STEPS,
           "generation_budgets": [GENERATIONS_LOW, GENERATIONS_HIGH], "offspring": ATTACK_OFFSPRING,
           "epsilon_scaled_l2": EPSILON, "seed": SEED}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
SNAPSHOT = SESSION / "source_snapshot"
for p in source_paths:
    target = SNAPSHOT / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    # Preserve a prior log if a cell is repeated.
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Fix the error, then rerun this cell.")

print("Notebook Python:", sys.version.split()[0])
print("Project commit:", project_commit)
print("Results folder:", SESSION)

# Log accelerator allocation without importing or changing Colab's PyTorch.
hardware = {"evaluation_device": "cpu", "platform": sys.platform,
            "reason": "Pinned constrained-attack pipeline is validated on CPU only."}
if shutil.which("nvidia-smi"):
    try:
        probe = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                                "--format=csv,noheader"], capture_output=True, text=True, timeout=15)
        hardware.update({"nvidia_smi_returncode": probe.returncode,
                         "allocated_gpu": probe.stdout.strip(), "probe_stderr": probe.stderr.strip()})
    except (subprocess.SubprocessError, OSError) as exc:
        hardware["probe_error"] = str(exc)
else:
    hardware["allocated_gpu"] = None
(SESSION / "hardware.json").write_text(json.dumps(hardware, indent=2) + "\n")
print("Evaluation device: CPU. Allocated GPU:", hardware.get("allocated_gpu") or "none detected")
print("An allocated GPU is not used by this pinned pilot. CPU runtime is sufficient.")


## 2. Isolated baseline environment

Python 3.8.20 and CPU PyTorch 1.12.1 run separately from Colab's notebook Python. The original pinned upstream code is preserved. No kernel restart or manual package installation is needed.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
shutil.copy2(ROOT / ".cache/setup_report.json", SESSION / "setup_report.json")
print("Notebook interpreter unchanged. Baseline interpreter:", PYTHON)


## 3. Verify dataset and pretrained checkpoints

Download approximately 348 MB on a fresh checkout. Existing files are checked against pinned SHA-256 hashes and reused when valid. These are precomputed CTU features, not PCAP traffic.


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
shutil.copy2(ROOT / "assets/download_manifest.json", SESSION / "download_manifest.json")


## 4. Run both diagnostic budgets

Each run scores the full native clean partition and then attacks the paired subset for both models. The logger saves candidate arrays, individual constraint checks and disjoint objective combinations. It adds computation and disk I/O, so its wall times are not inference-speed benchmarks or directly comparable to the uninstrumented pilot.

NoAttack is a control; CAPGD is the gradient stage; Moeva2 is the evolutionary stage. Already-fooled records may leave the attack sequence early. The stage output is not every candidate from every optimizer iteration.


In [ ]:
RUN_DIRS = {}
for generations in [GENERATIONS_LOW, GENERATIONS_HIGH]:
    print(f"Running candidate diagnostics: {generations} generations, both models.")
    run_dir = SESSION / unique_name(f"diagnostics32_g{generations}")
    RUN_DIRS[generations] = run_dir
    run_command([PYTHON, ROOT / "baseline/run_diagnostics.py", "--repo", ROOT / "vendor/tabularbench",
                 "--assets", ROOT / "assets", "--out", run_dir, "--attack",
                 "--n-attack", str(PILOT_RECORDS), "--steps", str(ATTACK_STEPS),
                 "--generations", str(generations), "--offspring", str(ATTACK_OFFSPRING),
                 "--eps", str(EPSILON), "--seed", str(SEED)], f"diagnostics_g{generations}.log")
    run_command([PYTHON, ROOT / "scripts/summarize_results.py", run_dir], f"summary_g{generations}.log")


## 5. Read the comparison

A successful candidate must satisfy **all three** conditions: valid features, distance within the fixed budget, and misclassification. Failure columns can overlap; the eight disjoint combinations in each `stage_summary.json` give an exact partition. New evasions are still scored only among records detected before attack.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

reports = {g: json.loads((p / "results.json").read_text()) for g, p in RUN_DIRS.items()}
low_report = reports[GENERATIONS_LOW]
ids = low_report["selection"]["global_csv_row_indices_zero_based"]
expected_counts = {"default": (54587, 88, 9, 398), "madry": (54633, 42, 11, 396)}
comparison, stages, checks = [], [], []
for generations, run_dir in RUN_DIRS.items():
    report = reports[generations]
    if report["selection"]["global_csv_row_indices_zero_based"] != ids:
        raise RuntimeError("Selected rows differ across search budgets.")
    for setting in ["models", "n_attack", "steps", "offspring", "eps", "seed"]:
        if report["settings"][setting] != low_report["settings"][setting]:
            raise RuntimeError(f"Uncontrolled change in {setting}.")
    for name in ["default", "madry"]:
        model = report["models"][name]
        actual = tuple(model["clean"][k] for k in ["tn", "fp", "fn", "tp"])
        checks.append({"generations": generations, "model": name, "clean_reference_matches": actual == expected_counts[name]})
        if actual != expected_counts[name]:
            print(f"Investigate clean-reference mismatch: {name}, generations={generations}, counts={actual}")
        with np.load(run_dir / f"{name}_attack_arrays.npz", allow_pickle=False) as data:
            if not np.array_equal(data["csv_rows"], ids):
                raise RuntimeError("Returned candidate rows differ from selected rows.")
            changed = int(np.any(data["x_adv"] != data["x_clean"], axis=1).sum())
        attack = model["attack"]
        comparison.append({"generations": generations, "model": name,
                           "initially_detected": attack["clean_detected"], "already_missed": attack["already_missed_before_attack"],
                           "returned_changed_rows": changed, "new_evasions": attack["newly_evaded"],
                           "recall_after_attack_percent": 100 * attack["malicious_recall_after_attack"],
                           "attack_plus_diagnostics_seconds": attack["attack_wall_seconds"]})
        trace = json.loads((run_dir / "diagnostics" / name / "stage_summary.json").read_text())
        for stage in trace["stages"]:
            if sum(b["candidates"] for b in stage["disjoint_objective_buckets"]) != stage["candidates"]:
                raise RuntimeError("Diagnostic buckets do not sum to candidate count.")
            stages.append({"generations": generations, "model": name, "stage": stage["stage"],
                           "inputs": stage["inputs"], "candidates": stage["candidates"],
                           "changed": stage["changed_candidates"], "changed_valid_in_budget": stage["changed_valid_in_budget"],
                           "fooled": stage["fooled_candidates"], "successful": stage["successful_candidates"],
                           "rows_with_success": stage["rows_with_successful_candidate"],
                           "invalid": stage["invalid_candidates"], "over_budget": stage["over_budget_candidates"],
                           "still_detected": stage["still_detected_candidates"],
                           **{f"fails_{k}": v for k, v in stage["constraint_failures_overlapping"].items()}})
comparison = pd.DataFrame(comparison)
stages = pd.DataFrame(stages)
comparison.to_csv(SESSION / "budget_comparison.csv", index=False)
stages.to_csv(SESSION / "candidate_stage_comparison.csv", index=False)
(SESSION / "comparison_checks.json").write_text(json.dumps({
    "same_selected_rows_across_models_and_budgets": True,
    "only_generation_budget_changed": True,
    "clean_reference_checks": checks,
    "timings_include_diagnostic_overhead": True,
    "scope": "Stage-output candidates before CAA selection; not every optimizer iteration."
}, indent=2) + "\n")
print("Final IDS outcomes on the same malicious records:")
display(comparison)
print("Stage-output diagnostics. Failure columns overlap; do not add them together:")
display(stages)
print("NoAttack is the unchanged-input control, not an attack success claim.")
print("Candidate counts are search outputs, not independent network attacks or requests.")
print("If the longer search also finds zero new evasions, retain the result; it does not prove security.")


## 6. Save and download the evidence

The ZIP includes results, timings, selected rows, candidate arrays, logs, hardware information, environment versions and a snapshot of the executed baseline scripts. It excludes the large source dataset and environment. Optional Drive copying prompts for authorization only when enabled.

**Share this ZIP**, even if no new evasions were found. If a cell fails, run this export cell separately to preserve partial logs. Colab runtime storage is temporary.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## Interpretation and next research decision

- **Changed candidates are invalid:** inspect the relation, bounds, type and immutable-feature checks. This identifies implementation or feasibility questions; it is not permission to relax the declared threat model.
- **Changed candidates are valid but over budget:** they do not count as successes at radius 0.5. Retain the predefined radius for this comparison.
- **Changed candidates are valid and in budget but still detected:** the search has not found an evasion. More generations measure one aspect of search effort; they cannot certify robustness.
- **Valid, in-budget evasions appear:** compare the two models' final new-evasion rates on the same subset. Extend the protocol before claiming a general defense advantage.

These are recorded CTU feature aggregates, not live packets. New training will require train-only preprocessing and a separate development/test protocol. Keep this baseline diagnostic as a reproducibility record rather than selecting thesis hyperparameters on these test rows.
